# 08 · Web Demo — upload leaf images from the browser  🌿

A **Gradio web app** for the evaluator. Unlike `google.colab.files.upload()` (which only works in the Colab browser, NOT the VS Code extension), this launches a real web page with a public **`*.gradio.live`** link that opens in any browser.

**How to run:** run all cells. The last cell prints a public URL — open it, then:
1. Upload a leaf image (drag-drop or file picker).
2. Optionally choose the true species from the dropdown.
3. Click **Classify** — see all three models' prediction, confidence, top-3 and Grad-CAM.
4. Keep uploading; if you set true species, a **running accuracy scoreboard** builds up across all your labelled uploads.

In [ ]:
# === Preamble 1/2: environment & GPU report ===
import sys
print('Python:', sys.version.split()[0])
try:
    import torch
    print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except ImportError:
    print('torch installs in the next cell.')

In [ ]:
# === Preamble 2/2: clone-or-pull + install (installs gradio too) ===
import os, subprocess, sys

REPO_URL = "https://github.com/Kidhurshan/plant-leaf-classifier.git"
REPO_DIR = "/content/plant-leaf-classifier"

if not os.path.isdir(REPO_DIR):
    print('Cloning', REPO_URL)
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# requirements.txt now includes gradio
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                'requirements.txt'], check=True)

from src.utils import sync_repo, gpu_report
sync_repo(); gpu_report()

## Load all three trained models (from checkpoints on Drive)

In [ ]:
from src.config import load_config
from src.utils import use_drive_paths
from src.inference import ModelComparer
import torch

cfg = load_config('configs/default.yaml')
use_drive_paths(cfg)                 # checkpoints live on Google Drive
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
comparer = ModelComparer(cfg, device)
print('Loaded models:', list(comparer.models))
print('Species      :', comparer.class_names)

## Launch the web app
Wait for the **public URL** (`Running on public URL: https://xxxx.gradio.live`) and open it in a browser. The link stays live while this cell runs.

In [ ]:
from src.webdemo import launch_demo
# share=True prints a public https://*.gradio.live link (needed on Colab).
launch_demo(comparer, share=True)

---
### Notes
* The public link is active only while the cell above is running. Stop the cell (or the runtime) to take the app offline.
* Uploaded images are processed in memory on the runtime; nothing is saved to the repo.
* **When finished:** `Runtime > Disconnect and delete runtime` — Colab compute units are consumed while a runtime is connected.